<a href="https://colab.research.google.com/github/thikhamporn0589-bit/Colab/blob/main/Lab2GE338final.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import ee
import geemap

ee.Authenticate()
ee.Initialize(project='ee-thikhamporn0589')

# =========================
# 📍 ROI
# =========================
thailand = ee.FeatureCollection("FAO/GAUL/2015/level2")

lampang = thailand.filter(ee.Filter.eq('ADM1_NAME', 'Lampang'))
roi = lampang.filter(ee.Filter.eq('ADM2_NAME', 'Muang Lampang'))
roi = ee.FeatureCollection([roi.first()])

# =========================
# 🛰️ Function scale
# =========================
def apply_scale_factors(image):
    opticalBands = image.select('SR_B.').multiply(0.0000275).add(-0.2)
    thermalBands = image.select('ST_B.*').multiply(0.00341802).add(149.0)
    return image.addBands(opticalBands, None, True)\
                .addBands(thermalBands, None, True)

# =========================
# 🌱 ต้นฤดูแล้ง (Nov–Dec)
# =========================
l8_early = (ee.ImageCollection("LANDSAT/LC08/C02/T1_L2")
            .filterBounds(roi)
            .filterDate('2023-11-01', '2023-12-31')
            .filter(ee.Filter.lt('CLOUD_COVER', 10))
            .map(apply_scale_factors)
            .median()
            .clip(roi))

# =========================
# 🔥 ปลายฤดูแล้ง (Mar–Apr)
# =========================
l8_late = (ee.ImageCollection("LANDSAT/LC08/C02/T1_L2")
           .filterBounds(roi)
           .filterDate('2024-03-01', '2024-04-30')
           .filter(ee.Filter.lt('CLOUD_COVER', 10))
           .map(apply_scale_factors)
           .median()
           .clip(roi))

# =========================
# 🌿 NDVI
# =========================
ndvi_early = l8_early.normalizedDifference(['SR_B5', 'SR_B4']).rename('NDVI_early')
ndvi_late = l8_late.normalizedDifference(['SR_B5', 'SR_B4']).rename('NDVI_late')

# =========================
# 💧 NDWI
# =========================
ndwi_early = l8_early.normalizedDifference(['SR_B3', 'SR_B5']).rename('NDWI_early')
ndwi_late = l8_late.normalizedDifference(['SR_B3', 'SR_B5']).rename('NDWI_late')

# =========================
# 🔥 Difference (สำคัญมาก)
# =========================
ndvi_diff = ndvi_late.subtract(ndvi_early).rename('NDVI_change')
ndwi_diff = ndwi_late.subtract(ndwi_early).rename('NDWI_change')

# =========================
# 🗺️ Map
# =========================
Map = geemap.Map()
Map.centerObject(roi, 11)

vis_ndvi = {'min': -1, 'max': 1, 'palette': ['blue', 'white', 'green']}
vis_diff = {'min': -0.5, 'max': 0.5, 'palette': ['red', 'white', 'green']}

# Early
Map.addLayer(ndvi_early, vis_ndvi, 'NDVI Early Dry')

# Late
Map.addLayer(ndvi_late, vis_ndvi, 'NDVI Late Dry')

# Change
Map.addLayer(ndvi_diff, vis_diff, 'NDVI Change')

Map.addLayer(roi, {'color': 'red'}, "Boundary", False)

Map

In [ ]:
districts = thailand.filter(ee.Filter.eq('ADM1_NAME', 'Lampang'))

In [ ]:
combined = ndvi_early.addBands(ndvi_late)\
                     .addBands(ndwi_early)\
                     .addBands(ndwi_late)

In [ ]:
zonal_stats = combined.reduceRegions(
    collection=districts,
    reducer=ee.Reducer.mean(),
    scale=30
)

In [ ]:
print(zonal_stats.limit(5).getInfo())

In [ ]:
def add_change(feature):
    ndvi_e = ee.Number(
        ee.Algorithms.If(feature.get('NDVI_early'), feature.get('NDVI_early'), 0)
    )

    ndvi_l = ee.Number(
        ee.Algorithms.If(feature.get('NDVI_late'), feature.get('NDVI_late'), 0)
    )

    ndwi_e = ee.Number(
        ee.Algorithms.If(feature.get('NDWI_early'), feature.get('NDWI_early'), 0)
    )

    ndwi_l = ee.Number(
        ee.Algorithms.If(feature.get('NDWI_late'), feature.get('NDWI_late'), 0)
    )

    return feature.set({
        'NDVI_change': ndvi_l.subtract(ndvi_e),
        'NDWI_change': ndwi_l.subtract(ndwi_e)
    })

zonal_stats = zonal_stats.map(add_change)

In [ ]:
task = ee.batch.Export.table.toDrive(
    collection=zonal_stats,
    description='Zonal_Full_Lampang',
    folder='GEE',
    fileNamePrefix='zonal_full',
    fileFormat='CSV'
)
task.start()

In [ ]:
Map.addLayer(
    zonal_stats,
    {},
    'Zonal Stats'
)

In [ ]:
# Visualization
vis_ndvi = {'min': -1, 'max': 1, 'palette': ['blue', 'white', 'green']}
vis_change = {'min': -0.5, 'max': 0.5, 'palette': ['red', 'white', 'green']}

Map = geemap.Map()
Map.centerObject(roi, 10)

# NDVI ต้นฤดูแล้ง
Map.addLayer(ndvi_early, vis_ndvi, 'NDVI Early Dry')

# NDVI ปลายฤดูแล้ง
Map.addLayer(ndvi_late, vis_ndvi, 'NDVI Late Dry')

# NDVI Change
Map.addLayer(ndvi_change, vis_change, 'NDVI Change')

Map.addLayer(roi, {'color': 'red'}, 'Boundary')

Map

In [ ]:
mean_early = ndvi_early.reduceRegion(
    ee.Reducer.mean(), roi, 30
)

mean_late = ndvi_late.reduceRegion(
    ee.Reducer.mean(), roi, 30
)

print("NDVI Early:", mean_early.getInfo())
print("NDVI Late:", mean_late.getInfo())

In [ ]:
# =========================
# 🔧 Function scale (ถ้ายังไม่มี)
# =========================
def scale(image):
    optical = image.select('SR_B.').multiply(0.0000275).add(-0.2)
    return image.addBands(optical, None, True)

# =========================
# 🌱 Early (ต้นฤดูแล้ง)
# =========================
early = (ee.ImageCollection("LANDSAT/LC08/C02/T1_L2")
         .filterBounds(roi)
         .filterDate('2023-11-01', '2023-12-31')
         .filter(ee.Filter.lt('CLOUD_COVER', 10))
         .map(scale)
         .median()
         .clip(roi))

# =========================
# 🔥 Late (ปลายฤดูแล้ง)
# =========================
late = (ee.ImageCollection("LANDSAT/LC08/C02/T1_L2")
        .filterBounds(roi)
        .filterDate('2024-03-01', '2024-04-30')
        .filter(ee.Filter.lt('CLOUD_COVER', 10))
        .map(scale)
        .median()
        .clip(roi))

In [ ]:
ndmi_early = early.normalizedDifference(['SR_B5', 'SR_B6']).rename('NDMI_early')
ndmi_late = late.normalizedDifference(['SR_B5', 'SR_B6']).rename('NDMI_late')

ndmi_change = ndmi_late.subtract(ndmi_early).rename('NDMI_change')

In [ ]:
vis_ndmi = {
    'min': -1,
    'max': 1,
    'palette': ['brown', 'white', 'blue']
}

Map = geemap.Map()
Map.centerObject(roi, 10)

Map.addLayer(ndmi_early, vis_ndmi, 'NDMI Early')
Map.addLayer(ndmi_late, vis_ndmi, 'NDMI Late')
Map.addLayer(ndmi_change, vis_ndmi, 'NDMI Change')

Map.addLayer(roi, {'color': 'red'}, 'Boundary')

Map

In [ ]:
mean_ndmi_early = ndmi_early.reduceRegion(
    ee.Reducer.mean(), roi, 30
)

mean_ndmi_late = ndmi_late.reduceRegion(
    ee.Reducer.mean(), roi, 30
)

print("NDMI Early:", mean_ndmi_early.getInfo())
print("NDMI Late:", mean_ndmi_late.getInfo())